# 3. HybridRAG & End-to-End Evaluation — Medical Textbook AlzRAGBench

**Part of the AlzRAGBench project** — Medical Textbook variant.

This notebook integrates **VectorRAG** (Notebook 2) and **GraphRAG** (Notebook 1) into a unified **HybridRAG** architecture, runs all three retrieval methods head-to-head on the 30-question evaluation set, and generates comparative metric plots.

## 3.1 Imports and Setup

In [ ]:
import os
import sys
import json
import time
import pickle
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from google import genai

# Paths
NOTEBOOK_DIR = Path(".").resolve()
BASE_DIR     = NOTEBOOK_DIR.parent
DATASET_DIR  = BASE_DIR / "Dataset"
NODES_CSV    = DATASET_DIR / "Knowledge graph" / "nodes.csv"
EDGES_CSV    = DATASET_DIR / "Knowledge graph" / "edges.csv"
CHUNKS_PATH  = DATASET_DIR / "chunking" / "_all_chunks.json"
EVAL_PATH    = DATASET_DIR / "Evaluation" / "eval_dataset.json"
VECTOR_DIR   = BASE_DIR / "vector_output"
RESULTS_DIR  = BASE_DIR / "Results"
RESULTS_DIR.mkdir(exist_ok=True)

# Gemini API Setup
load_dotenv(BASE_DIR / ".env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Please set GEMINI_API_KEY in .env"
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-flash-lite-latest"

print("Setup complete.")

## 3.2 Load Models, Indexes, and Graph

In [ ]:
# Load Graph
nodes_df = pd.read_csv(NODES_CSV)
edges_df = pd.read_csv(EDGES_CSV)
G = nx.Graph()
for _, r in nodes_df.iterrows():
    G.add_node(r["nodeId:ID"], name=r["name"], label=r["type:LABEL"], description=r["description"])
for _, r in edges_df.iterrows():
    G.add_edge(r[":START_ID"], r[":END_ID"], relation=r[":TYPE"], evidence=r["evidence"])

# Load Vector Index
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
faiss_index = faiss.read_index(str(VECTOR_DIR / "faiss.index"))
with open(VECTOR_DIR / "chunk_texts.pkl", "rb") as f: texts = pickle.load(f)
with open(VECTOR_DIR / "metadata.pkl", "rb") as f: metadata = pickle.load(f)

print(f"Graph Nodes: {G.number_of_nodes()} | Vector Chunks: {faiss_index.ntotal}")

## 3.3 RAG Retrieval Functions

In [ ]:
def vector_get_context(query, top_k=5):
    q_emb = EMBED_MODEL.encode([query], convert_to_numpy=True).astype("float32")
    dists, idxs = faiss_index.search(q_emb, top_k)
    return "\n\n".join([texts[i] for i in idxs[0] if i != -1])

def graph_get_context(query):
    q = query.lower()
    matched = [n for n, d in G.nodes(data=True) if q in str(d.get("name","")).lower() or q in str(d.get("description","")).lower()]
    if not matched: return ""
    ctx = ""
    visited = set()
    for node in matched:
        if node in visited: continue
        visited.add(node)
        nd = G.nodes[node]
        ctx += f"Entity: {nd.get('name','')}\nType: {nd.get('label','')}\nDesc: {nd.get('description','')}\nRelations:\n"
        for nbr in G.neighbors(node):
            rel = G.get_edge_data(node, nbr).get('relation','')
            ctx += f"  -- {rel} --> {G.nodes[nbr].get('name', nbr)}\n"
        ctx += "\n"
    return ctx

def hybrid_get_context(query):
    v_ctx = vector_get_context(query, top_k=5)
    g_ctx = graph_get_context(query)
    return f"========================\nTEXTBOOK KNOWLEDGE\n========================\n{v_ctx}\n\n========================\nKNOWLEDGE GRAPH\n========================\n{g_ctx}"

def llm_generate(question, context):
    prompt = f"Answer strictly using context:\n{context}\n\nQuestion: {question}"
    time.sleep(4) # Throttling for free tier API
    res = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    return res.text

## 3.4 Evaluation Metrics & Benchmark Execution

In [ ]:
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def score_rouge(expected, predicted):
    return rouge.score(expected, predicted)["rougeL"].fmeasure

def score_similarity(expected, predicted):
    e1 = EMBED_MODEL.encode([expected], convert_to_numpy=True)
    e2 = EMBED_MODEL.encode([predicted], convert_to_numpy=True)
    return float(cosine_similarity(e1, e2)[0][0])

ABLATION_CSV = RESULTS_DIR / "ablation_results.csv"

if ABLATION_CSV.exists():
    print(f"Loading results from {ABLATION_CSV}")
    df_results = pd.read_csv(ABLATION_CSV)
else:
    with open(EVAL_PATH, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    questions = eval_data["questions"]
    results = []
    
    for i, item in enumerate(questions):
        q = item["question"]
        gold = item["expected_answer"]
        fav = item["designed_to_favor"]
        
        # VectorRAG
        v_ctx = vector_get_context(q)
        v_ans = llm_generate(q, v_ctx)
        
        # GraphRAG
        g_ctx = graph_get_context(q)
        g_ans = llm_generate(q, g_ctx)
        
        # HybridRAG
        h_ctx = hybrid_get_context(q)
        h_ans = llm_generate(q, h_ctx)
        
        for m, a in [("VectorRAG", v_ans), ("GraphRAG", g_ans), ("HybridRAG", h_ans)]:
            results.append({
                "question_id": item["question_id"],
                "question": q,
                "designed_to_favor": fav,
                "method": m,
                "expected_answer": gold,
                "generated_answer": a,
                "rouge_l": score_rouge(gold, a),
                "similarity": score_similarity(gold, a)
            })
            
    df_results = pd.DataFrame(results)
    df_results.to_csv(ABLATION_CSV, index=False)

## 3.5 Summary & Visualisation

In [ ]:
# Load ablation results
df_results = pd.read_csv(RESULTS_DIR / "ablation_results.csv")

summary = df_results.groupby("method")[["similarity", "rouge_l"]].mean().round(4).reset_index()
summary.to_csv(RESULTS_DIR / "summary_table.csv", index=False)
print("Mean Performance Summary:")
print(summary.to_string(index=False))

# Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
methods = ["VectorRAG", "GraphRAG", "HybridRAG"]
colors = ["#4C9BE8", "#E8804C", "#4CE8A0"]

sim_vals = [df_results[df_results.method==m]["similarity"].mean() for m in methods]
rouge_vals = [df_results[df_results.method==m]["rouge_l"].mean() for m in methods]

ax[0].bar(methods, sim_vals, color=colors)
ax[0].set_title("Semantic Similarity")
ax[0].set_ylim(0, 1)

ax[1].bar(methods, rouge_vals, color=colors)
ax[1].set_title("ROUGE-L Score")
ax[1].set_ylim(0, 1)

plt.suptitle("AlzRAGBench — Medical Textbook: RAG Method Comparison", fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ablation_results.png", dpi=150)
plt.show()